# Broadway Show Finder: Build the Vector Database
Run this notebook **once** on your local machine. It creates a `chromadb/` folder, which `app.py` loads later.

Before the first run, install the required libraries in a terminal (run it in the same folder as `app.py`):
```
pip install -r requirements.txt
pip install gradio
```

## 1. Load the data

In [1]:
import pandas as pd

url = "https://raw.githubusercontent.com/rfordatascience/tidytuesday/main/data/2020/2020-04-28/synopses.csv"
df = pd.read_csv(url)

print("Original rows:", len(df))
df.head()

Original rows: 1122


,show,synopsis
0,42nd Street,"A chorus girl from Allentown, PA, steps into t..."
1,A Chorus Line,Broadway gypsies bare their souls and put thei...
2,Aren't We All?,English nobility seek out marriage proposals —...
3,Arms and the Man,NaN
4,As Is,NaN


## 2. Clean the data

In [2]:
# Drop rows with no synopsis
df = df.dropna(subset=["synopsis"])

# Remove extra spaces at the start and end
df["synopsis"] = df["synopsis"].str.strip()

# Keep only one row per show
df = df.drop_duplicates(subset=["show"])

# Drop very short synopses (under 20 characters carry little information)
df = df[df["synopsis"].str.len() >= 20]

print("Rows after cleaning:", len(df))

Rows after cleaning: 945


## 3. Chunking
The synopses are short (about 200+ characters on average), so **one row = one chunk**. No further splitting is needed.

In [3]:
texts = df["synopsis"].tolist()   # text to embed
names = df["show"].tolist()       # show names, displayed in search results

print(names[0])
print(texts[0])

42nd Street
A chorus girl from Allentown, PA, steps into the starring role to save the show when its star breaks her leg, in this stage adaptation of the classic 1933 film musical.


## 4. Generate embeddings

In [4]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
embeddings = model.encode(texts, show_progress_bar=True)

print(embeddings.shape)   # (number of shows, 384)

/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1117 13:19:24.774000 65360 site-packages/torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.
Batches: 100%|██████████| 30/30 [00:01<00:00, 17.92it/s]

(945, 384)


## 5. Store in a Chroma database (saved to the `chromadb/` folder)

In [5]:
import chromadb

client = chromadb.PersistentClient(path="chromadb")

# If a collection with this name already exists, delete it to avoid duplicates
try:
    client.delete_collection("broadway")
except Exception:
    pass

# "cosine" means vectors are compared with cosine similarity
collection = client.create_collection(name="broadway", metadata={"hnsw:space": "cosine"})

collection.add(
    ids=[str(i) for i in range(len(texts))],
    documents=texts,
    embeddings=embeddings.tolist(),
    metadatas=[{"show": name} for name in names]
)

print("Number of shows in the database:", collection.count())

Number of shows in the database: 945


## 6. Test the search

In [6]:
def search(query, n=3):
    query_embedding = model.encode([query]).tolist()
    results = collection.query(query_embeddings=query_embedding, n_results=n)

    for i in range(n):
        name = results["metadatas"][0][i]["show"]
        text = results["documents"][0][i]
        score = 1 - results["distances"][0][i]   # cosine distance -> similarity
        print(f"{i+1}. {name}  (similarity: {score:.2f})")
        print("  ", text)
        print()

search("a love story in Paris")

1. Cirque du Soleil Paramour  (similarity: 0.52)
   Set in the glamorous world of Golden Age Hollywood, this groundbreaking event from Cirque du Soleil spins the tale of a beautiful young poet forced to choose between love and art.

2. The Height of the Storm  (similarity: 0.51)
   André and Madeleine have been in love for over fifty years. This weekend, as their daughters visit, something feels unusual. A bunch of flowers arrives, but who sent them? A woman from the past turns up, but who is she? And why does André feel like he isn’t there at all?

3. The Nance  (similarity: 0.49)
   The Nance tells the story of Chauncey Miles, a headline nance performer in New York burlesque, who also happens to be a homosexual. Playwright Douglas Carter Beane paints the portrait of a homosexual man living and working in the secretive and dangerous gay world of 1930s New York.



In [7]:
search("a funny show about a family")

1. You Can't Take It With You  (similarity: 0.56)
   An eccentric family allows each member to pursue his own ambition in the home, but their routine is disrupted when a daughter with a regular job wants to bring her fiancé and his Wall Street family to dinner in George S. Kaufman and Moss Hart's Pulitzer Prize-winning comedy.

2. The Royal Family  (similarity: 0.50)
   In this thinly veiled parody of the Barrymore family circa 1927, a flamboyant theatrical family must cope with the dumbfounding fact that one of them wishes to marry someone outside of show business.

3. Tartuffe  (similarity: 0.49)
   A religious conman infiltrates the household of a gullible man and his exasperated family in Moliere's 1664 comedy.



In [8]:
search("historical story about American revolution")

1. The Devil's Disciple  (similarity: 0.43)
   A town drunkard substitutes himself for the minister British soldiers are attempting to capture and hang, in George Bernard Shaw's 1897 satire of New England Puritanism and political self-importance during the Revolutionary War.

2. Amazing Grace  (similarity: 0.43)
   Based on the awe-inspiring true story behind the world’s most beloved song, Amazing Grace tells the story of John Newton whose incredible journey ignited a historic wave of change that gave birth to the abolitionist movement.

3. Bloody Bloody Andrew Jackson  (similarity: 0.41)
   A mix of historical fact and invention, Bloody Bloody Andrew Jackson uses the story of America's controversial seventh president to investigate the attraction and terrors of American populism, using a blend of outrageous comedy, anarchic theatricality and infectious emo rock.



## Next steps
- Check that a `chromadb/` folder now exists in this folder
- Run `python app.py` and test the app in your browser
- Upload `app.py`, `requirements.txt`, and the entire `chromadb/` folder to your Hugging Face Space